In [1]:
# Mount Google Drive
from google.colab import drive
#drive.mount('/content/drive',force_remount=True)
drive.mount('/content/drive')

Mounted at /content/drive


# **LightGBM and Statistical Benchmark models for Andrade & Cunha paper on disaggregated XGB (2023)**



# Reference:
* Andrade & Cunha (2023). Disaggregated retail demand forecasting:A gradient boosting approach
*  https://doi.org/10.1016/j.asoc.2023.110283

* Dataset:    Corporación Favorita (Kaggle), weekly granularity
* Series:     1,623 items × 54 stores (items used in the paper's pipeline)
* Train:      All weeks before the final 8 weeks (start date varies per store)
* Test:       Last 8 weeks : June 25 to August 13 2017
* Evaluation: h=1 through h=8 weekly horizons

# Statistical models - AutoETS for Smooth category and SBA for non-smooth
* ADI threshold of 1.32 is used to classify series into smooth and non-smooth
* Smooth is modelled using AutoETS which automatically determines best model using AICc (seaosnl vs non-seasonal)
* non-smooth is modelled using SBA (crostonSBA)

# LGBM model: Per-store models (54 models) matching the paper's XGBoost design
* Uses the paper's precomputed lagged_features_2.csv feature set
* seed=42 added for reproducibility (paper's code has no seed)
* Statistical: SBC at weekly granularity. AutoETS(season_length=52) captures the annual seasonal cycle.

# Prerequisites:
Run the paper's Articles 1–5  from the paper's repository (CodeOcean capsule 3786508) to generate the following files:
* calculated_features/lagged_features_2.csv
* calculated_features/product_features.csv
* calculated_features/store_features.csv
* calculated_features/seasonal_features.csv
* calculated_features/features/calendar_features.csv

# Required packages: lightgbm, statsforecast, pandas, numpy, scikit-learn
* do a pip install lightgbm statsforecast pandas numpy scikit-learn

**Import libraries**

In [2]:
!pip install lightgbm statsforecast pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.0/501.0 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 6.8 MB/s eta 0:00:00


In [3]:
##import required packages

import os
import glob
import gc
import time
import logging
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, CrostonOptimized, CrostonSBA
from sklearn.metrics import r2_score

#suppress warnings and enable logging
warnings.filterwarnings("ignore")


**Set paths (change as needed)**

In [4]:
##### paths #######################################
DATA_DIR    = "/content/drive/MyDrive/tll_reproducibility/" #replace with your path
RESULTS_DIR = "/content/drive/MyDrive/tll_reproducibility/disaggregated_xgb_paper/xgb_results" #replace with your path
STORE_DIR   = os.path.join(RESULTS_DIR, "lgbm_store_results")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(STORE_DIR,   exist_ok=True)

# logging — file handler plus console. force=True is required in Colab
LOG_PATH = os.path.join(RESULTS_DIR, "run_log.txt")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.FileHandler(LOG_PATH, mode="a"), logging.StreamHandler()],
    force=True,
)
logger = logging.getLogger(__name__)
logger.info(f"Logging to {LOG_PATH}")

08:35:33  Logging to /content/drive/MyDrive/tll_reproducibility/disaggregated_xgb_paper/xgb_results/run_log.txt


**Configuration (parameters)**

In [5]:

########## config #######
TEST_WEEKS         = 8      # last 8 weeks held out as test
PERISHABLE_WEIGHT  = 1.25
SEED               = 42     # paper has no seed; 42 used for reproducibility
ADI_THRESH         = 1.32  #Syntetos & Boylan (2005)
MIN_OBS = 2 #crash guard against errors from statsForecast functions - minimum positive demand observations for the fit


np.random.seed(SEED)

#LGBM params based on XGB params from original paper
LGBM_PARAMS = {
    "objective"        : "regression",
    "metric"           : "rmse",
    "learning_rate"    : 0.1,
    "num_leaves"       : 31,
    "max_depth"        : 5,
    "colsample_bytree" : 0.3,
    "verbose"          : -1,
    "seed"             : SEED,
    "n_jobs"           : -1,
}
NUM_ROUNDS     = 100
EARLY_STOPPING = 20

In [6]:
##### load data ##################

logger.info("Loading feature files...")

t0 = time.time() #keep track of time

feat_base = os.path.join(DATA_DIR, "calculated_features")

#main feature file: lagges sales + rolling stats
dataset = pd.read_csv(os.path.join(feat_base, "lagged_features_2.csv"), header=[0, 1])

# collapse '105574_lag_1' to '105574' so xs(item, level=1) picks ALL blocks
for col in sorted(set(dataset.columns.get_level_values(0).tolist())):
    if col not in ["date", "store_nbr"]:
        d = {el: el.split("_")[0] for el in dataset[col].columns}
        dataset.rename(columns=d, level=1, inplace=True)

dataset = dataset.replace({'False': 0, 'True': 1, False: 0, True: 1})
dataset = dataset.infer_objects(copy=False)

#auxiliary feature files used
dataset_calendar = pd.read_csv(os.path.join(feat_base, "calendar_features.csv"))
df_products      = pd.read_csv(os.path.join(feat_base, "product_features.csv"))
df_stores_cat    = pd.read_csv(os.path.join(feat_base, "store_features.csv"))
df_seasonal      = pd.read_csv(os.path.join(feat_base, "seasonal_features.csv"))

logger.info(f"  Dataset shape: {dataset.shape}  ({(time.time()-t0):.1f}s)")
#raw items file for fetching perishable weights
df_items_raw = pd.read_csv(os.path.join("/content/drive/MyDrive/tll/raw_data", "items.csv"))

logger.info(f"lagged_features_2.csv  : {dataset.shape}  ({time.time()-t0:.1f}s)")
logger.info(f"product_features.csv   : {df_products.shape}")
logger.info(f"store_features.csv     : {df_stores_cat.shape}")
logger.info(f"seasonal_features.csv  : {df_seasonal.shape}")
logger.info(f"calendar_features.csv  : {dataset_calendar.shape}")

# stores and items from multi-index columns
stores    = dataset["store_nbr"].iloc[:, 0].unique()
meta_cols = ["date", "store_nbr"]
item_cols = [c for c in dataset.columns.get_level_values(1).unique()
             if str(c).isdigit()]   # keep only numeric item numbers
logger.info(f"  Stores: {len(stores)}  Items: {len(item_cols):,}")

# static feature lookup dicts for LGBM

product_cat_index = {
    int(row["item_nbr"]): df_products[
        df_products.item_nbr == row["item_nbr"]
    ].drop("item_nbr", axis=1).values
    for _, row in df_products.iterrows()
}
store_cat_index = {
    int(row["store_nbr"]): df_stores_cat[
        df_stores_cat.store_nbr == row["store_nbr"]
    ].drop("store_nbr", axis=1).values
    for _, row in df_stores_cat.iterrows()
}


#get weights
def get_weight(item_id):
    """Return perishable weight for an item (1.25 if perishable, else 1.0)"""
    row = df_items_raw[df_items_raw.item_nbr == item_id]
    return PERISHABLE_WEIGHT if (len(row) > 0 and row.iloc[0]["perishable"] == 1) else 1.0


08:35:34  Loading feature files...
08:36:35    Dataset shape: (9077, 27593)  (61.7s)
08:36:36  lagged_features_2.csv  : (9077, 27593)  (62.6s)
08:36:36  product_features.csv   : (4100, 36)
08:36:36  store_features.csv     : (54, 23)
08:36:36  seasonal_features.csv  : (177, 5)
08:36:36  calendar_features.csv  : (177, 66)
08:36:36    Stores: 54  Items: 1,623


In [7]:
print(dataset.shape,df_items_raw.shape, df_seasonal.shape, dataset_calendar.shape, df_products.shape, df_stores_cat.shape, dataset.shape)


(9077, 27593) (4100, 4) (177, 5) (177, 66) (4100, 36) (54, 23) (9077, 27593)


**Prepare train/test data splits per store**

In [8]:
####### prepare train/test split per store #######################

def get_store_data(store_id):
    """Extract and split data for one store. Returns train/test arrays and item list"""
    mask     = dataset["store_nbr"].iloc[:, 0] == store_id
    df_store = dataset[mask].copy()
    n_rows   = len(df_store)
    n_train  = n_rows - TEST_WEEKS

    X_all, y_all, w_all, items = [], [], [], []
    for item in item_cols:
        df_item = df_store.xs(item, axis=1, level=1)
        y_col     = df_item.columns[0] #unit_sales corrected
        feat_cols = df_item.columns[1:] #lag and rolling features
        X_all.append(df_item[feat_cols].values.astype(np.float32))
        y_all.append(df_item[y_col].values.astype(np.float32))
        w_all.append(np.full(n_rows, get_weight(int(item)), dtype=np.float32))
        items.append(int(item))

    X_all = np.vstack(X_all)
    y_all = np.concatenate(y_all)
    w_all = np.concatenate(w_all)
    n_items = len(items)

    # rows are ordered: [all items for week 1, all items for week 2, ...]
    train_idx = np.concatenate([np.arange(i * n_rows, i * n_rows + n_train) for i in range(n_items)])
    test_idx  = np.concatenate([np.arange(i * n_rows + n_train, (i+1) * n_rows) for i in range(n_items)])

    return X_all[train_idx], y_all[train_idx], X_all[test_idx], y_all[test_idx], w_all[test_idx], items

**LGBM features build**

In [9]:
def build_lgbm_features(store_id):
    mask            = dataset["store_nbr"].iloc[:, 0] == store_id
    store_dataset   = dataset[mask].copy()
    feature_dataset = store_dataset.drop("unit_sales_corrected", axis=1)
    target_dataset  = store_dataset[["unit_sales_corrected"]]
    n_samples       = store_dataset.shape[0]

    delta         = df_seasonal.shape[0] - n_samples
    seasonal_arr  = df_seasonal.iloc[delta:].values.astype(np.float32)
    calendar_arr  = dataset_calendar.iloc[delta:].values.astype(np.float32)
    # ALL items' features except target, date, store_nbr
    others_arr    = feature_dataset.drop(
        ["date", "store_nbr"], axis=1).values.astype(np.float32)
    store_cat_arr = np.tile(
        store_cat_index[int(store_id)], (n_samples, 1)).astype(np.float32)

    result = {}
    for item_nbr in target_dataset["unit_sales_corrected"].columns:
        own_arr     = feature_dataset.xs(
            item_nbr, axis=1, level=1, drop_level=False
        ).values.astype(np.float32)
        product_cat = np.tile(
            product_cat_index[int(item_nbr)], (n_samples, 1)).astype(np.float32)
        X = np.hstack([own_arr, others_arr, seasonal_arr,
                       calendar_arr, store_cat_arr, product_cat])
        y = target_dataset["unit_sales_corrected"][item_nbr].values.astype(np.float32)
        result[item_nbr] = (X, y)

    return result

In [10]:
# pf = build_lgbm_features(stores[0])
# X, y = pf[list(pf.keys())[0]]
# print(f"n_features: {X.shape[1]}")  # 26102

In [11]:
# import pandas as pd, numpy as np

# ds  = pd.read_csv(f"{feat_base}/lagged_features_2.csv", header=[0,1])
# tgt = ds["unit_sales_corrected"]
# store_col = ds["store_nbr"].iloc[:, 0]

# late, total = 0, 0
# med = []
# for s in store_col.unique():
#     blk = tgt[store_col == s].reset_index(drop=True)
#     has = (blk > 0).any()
#     f   = (blk > 0).idxmax()[has]
#     med.append(f.median())
#     late  += (f > 20).sum()
#     total += has.sum()

# print(f"weeks per store          : {(store_col == store_col.iloc[0]).sum()}")
# print(f"item x store series      : {total:,}")
# print(f"median first-sale week   : {np.median(med):.0f}")
# print(f"series starting > week 20: {late:,} ({late/total*100:.1f}%)")

**Statistical Benchmarks: SBC classification**

In [12]:
###### demand classification and statistical models ##################

logger.info("Demand classification (smooth/non-smooth) and building statistical data...")
t0 = time.time()

sbc_rows  = []
stat_rows = []

for store_id in stores:
    X_tr, y_tr, _, _, _, items = get_store_data(store_id)
    n_train = len(y_tr) // len(items)

    for k, item_id in enumerate(items):
        series_train = y_tr[k * n_train : (k + 1) * n_train]
        uid          = f"{store_id}_{item_id}"

        y_orig = np.expm1(np.array(series_train, dtype=float))
        nz     = y_orig[y_orig > 0]
        n_nz   = len(nz)

        if n_nz == 0:
            cat = "non-smooth"   # ADI = inf → non-smooth → SBA
        else:
            adi = len(y_orig) / n_nz
            cat = "smooth" if adi < ADI_THRESH else "non-smooth"

        sbc_rows.append({"unique_id": uid, "store_nbr": store_id,
                         "item_nbr": item_id, "category": cat})
        for t, val in enumerate(y_orig):
            stat_rows.append({"unique_id": uid, "ds": t, "y": max(0.0, float(val))})

df_sbc       = pd.DataFrame(sbc_rows)
df_stat_long = pd.DataFrame(stat_rows)
df_sbc.to_csv(os.path.join(RESULTS_DIR, "sbc_classification.csv"), index=False)
logger.info(f"  Done in {(time.time()-t0)/60:.1f} min")
logger.info(f"  Distribution:\n{df_sbc['category'].value_counts().to_string()}")


08:36:56  Demand classification (smooth/non-smooth) and building statistical data...
08:37:39    Done in 0.7 min
08:37:39    Distribution:
category
smooth        63206
non-smooth    24436


In [14]:
df_sbc['category'].value_counts()
# print((52579 + 10627)/df_sbc['category'].value_counts().sum(), (16672)/df_sbc['category'].value_counts().sum())

,count
category,
smooth,63206
non-smooth,24436


In [15]:
# print(type(dataset.columns.get_level_values(1)[5]))
# print(dataset.columns.get_level_values(1)[:10].tolist())

**Statistical Benchmarks : Models**

In [29]:
######## statistical models #######################################
# AutoETS(season_length=52) performs an internal model search across error,
# trend, and seasonal components and selects the specification minimizing
# its own AICc.It does not force seasonality just because season_length>1,
# and returns a non-seasonal ETS specification where seasonality doesn't
# improve the fit.CrostonSBA is the Syntetos-Boylan bias-corrected Croston variant.

logger.info("Fitting statistical models...")
t0 = time.time()

#get ETS model spec
def is_seasonal(method_str):
    """ETS(E,T,S) notation"""
    return method_str.split(",")[-1].rstrip(")") != "N"

def fit_stat(df_long, ids, model, label, h, min_obs=2):
    if not ids:
        return pd.DataFrame(columns=["unique_id", "ds", "pred"])
    df_sub    = df_long[df_long.unique_id.isin(ids)]
    n_obs     = df_sub.groupby("unique_id")["y"].count()
    valid_ids = n_obs[n_obs >= min_obs].index.tolist()
    tiny_ids  = n_obs[n_obs <  min_obs].index.tolist()
    logger.info(f"  {label}: {len(valid_ids):,} fitted  {len(tiny_ids):,} mean fallback")
    results = []
    if valid_ids:
        sf  = StatsForecast(models=[model], freq=1, n_jobs=-1)
        fc  = sf.forecast(df=df_sub[df_sub.unique_id.isin(valid_ids)], h=h).reset_index(drop=True)
        col = [c for c in fc.columns if c not in ["unique_id", "ds"]][0]
        fc  = fc.rename(columns={col: "pred"})[["unique_id", "ds", "pred"]]
        fc["pred"] = fc["pred"].clip(lower=0)
        results.append(fc)
    if tiny_ids:
        means = df_sub[df_sub.unique_id.isin(tiny_ids)].groupby("unique_id")["y"].mean()
        rows  = [{"unique_id": uid, "ds": int(n_obs.get(uid, 0)) + d,
                  "pred": max(0.0, float(means.get(uid, 0.0)))}
                 for uid in tiny_ids for d in range(h)]
        results.append(pd.DataFrame(rows))
    return pd.concat(results, ignore_index=True) if results else \
           pd.DataFrame(columns=["unique_id", "ds", "pred"])

# smooth (ADI < 1.32) → AutoETS(s=52), internally selecting seasonal or non-seasonal
# non-smooth (ADI ≥ 1.32) → CrostonSBA
smooth_ids     = df_sbc[df_sbc.category == "smooth"]["unique_id"].tolist()
non_smooth_ids = df_sbc[df_sbc.category == "non-smooth"]["unique_id"].tolist()

sf_ets = StatsForecast(models=[AutoETS(season_length=52)], freq=1, n_jobs=-1)
sf_ets.fit(df=df_stat_long[df_stat_long.unique_id.isin(smooth_ids)])
methods = {uid: sf_ets.fitted_[i, 0].model_["method"] for i, uid in enumerate(sf_ets.uids)}
selected_model = pd.Series(methods).apply(lambda m: "AutoETS_seas" if is_seasonal(m) else "AutoETS_ns")
logger.info(f"  AutoETS(s=52): {(selected_model=='AutoETS_seas').sum():,} seasonal, "
            f"{(selected_model=='AutoETS_ns').sum():,} non-seasonal")

fc_ets = fit_stat(df_stat_long, smooth_ids, AutoETS(season_length=52), "AutoETS(s=52)", TEST_WEEKS, min_obs=MIN_OBS)
fc_sba = fit_stat(df_stat_long, non_smooth_ids, CrostonSBA(),"CrostonSBA", TEST_WEEKS, min_obs=MIN_OBS)

df_stat_fc = pd.concat([fc_ets, fc_sba], ignore_index=True)
df_stat_fc.to_csv(os.path.join(RESULTS_DIR, "statistical_predictions.csv"), index=False)

pd.DataFrame({"unique_id": methods.keys(), "method": methods.values(),
              "selected_model": selected_model.values}).to_csv(
    os.path.join(RESULTS_DIR, "autoets_model_selection.csv"), index=False)

logger.info(f"  Statistical models done in {(time.time()-t0)/60:.1f} min")

10:11:33  Fitting statistical models...
10:14:34    AutoETS(s=52): 1,750 seasonal, 61,456 non-seasonal
10:14:35    AutoETS(s=52): 63,206 fitted  0 mean fallback
10:17:26    CrostonSBA: 24,436 fitted  0 mean fallback
10:17:32    Statistical models done in 6.0 min


In [30]:
df_stat_fc.head()

,unique_id,ds,pred
0,10_1003679,169,18.361230
1,10_1003679,170,12.627275
2,10_1003679,171,10.567464
3,10_1003679,172,9.767274
4,10_1003679,173,19.692772


In [31]:
fs = df_stat_long[df_stat_long.y > 0].groupby("unique_id")["ds"].min()
print(f"median first-sale week: {fs.median():.0f}  |  after week 20: {(fs > 20).mean()*100:.1f}%")

median first-sale week: 0  |  after week 20: 17.3%


In [32]:
# #clear the cached files from previous runs
# import glob, os
# old_files = glob.glob(os.path.join(STORE_DIR, "lgbm_store_*.csv"))
# for f in old_files:
#     os.remove(f)
# logger.info(f"Cleared {len(old_files)} cached store files — will retrain with full features")

**Machine Learning Benchmark: LGBM per store model**

In [33]:
##### LGBM per store ##################

logger.info("Training LightGBM per store...")
all_store_results = []

for s_idx, store_id in enumerate(stores):
    save_path = os.path.join(STORE_DIR, f"lgbm_store_{store_id}.csv")
    if os.path.exists(save_path):
        logger.info(f"Store {store_id}: loaded from cache")
        all_store_results.append(pd.read_csv(save_path))
        continue

    t0               = time.time()
    product_features = build_lgbm_features(store_id)
    item_list        = list(product_features.keys())

    # stack training data across all items (exclude last TEST_WEEKS)
    X_tr_list, y_tr_list = [], []
    for item_nbr, (X, y) in product_features.items():
        X_tr_list.append(X[:-TEST_WEEKS])
        y_tr_list.append(y[:-TEST_WEEKS])
    X_store = np.vstack(X_tr_list).astype(np.float32)
    y_store = np.concatenate(y_tr_list).astype(np.float32)
    del X_tr_list, y_tr_list

    # val split: last 10% of training or 512 rows minimum
    val_size    = max(512, int(0.1 * len(X_store)))
    X_tr, X_va = X_store[:-val_size], X_store[-val_size:]
    y_tr, y_va = y_store[:-val_size], y_store[-val_size:]
    del X_store, y_store; gc.collect()

    train_set = lgb.Dataset(X_tr, label=y_tr)
    val_set   = lgb.Dataset(X_va, label=y_va, reference=train_set)
    model_lgb = lgb.train(
        LGBM_PARAMS, train_set,
        num_boost_round=NUM_ROUNDS,
        valid_sets=[val_set],
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )
    del X_tr, X_va, y_tr, y_va; gc.collect()

    # predict test period per item and collect results
    store_rows = []
    for item_nbr in item_list:
        X, y      = product_features[item_nbr]
        preds     = np.expm1(np.clip(model_lgb.predict(X[-TEST_WEEKS:]), 0, None))
        y_te_orig = np.expm1(y[-TEST_WEEKS:])
        wt        = get_weight(int(item_nbr))
        for d in range(TEST_WEEKS):
            store_rows.append({
                "unique_id": f"{store_id}_{item_nbr}",
                "store_nbr": store_id,
                "item_nbr" : item_nbr,
                "horizon"  : d + 1,
                "actual"   : float(y_te_orig[d]),
                "pred"     : float(preds[d]),
                "weight"   : wt,
            })

    df_store = pd.DataFrame(store_rows)
    df_store.to_csv(save_path, index=False)
    all_store_results.append(df_store)
    logger.info(f"Store {store_id} ({s_idx+1}/{len(stores)}): "
                f"best_iter={model_lgb.best_iteration}  t={time.time()-t0:.1f}s")
    del model_lgb, product_features; gc.collect()

df_lgbm = pd.concat(all_store_results, ignore_index=True)
df_lgbm.to_csv(os.path.join(RESULTS_DIR, "lgbm_predictions.csv"), index=False) #save results

10:17:46  Training LightGBM per store...
10:17:46  Store 1: loaded from cache
10:17:46  Store 2: loaded from cache
10:17:46  Store 3: loaded from cache
10:17:46  Store 4: loaded from cache
10:17:46  Store 5: loaded from cache
10:17:46  Store 6: loaded from cache
10:17:46  Store 7: loaded from cache
10:17:46  Store 8: loaded from cache
10:17:46  Store 9: loaded from cache
10:17:46  Store 10: loaded from cache
10:17:46  Store 11: loaded from cache
10:17:46  Store 12: loaded from cache
10:17:46  Store 13: loaded from cache
10:17:46  Store 14: loaded from cache
10:17:46  Store 15: loaded from cache
10:17:46  Store 16: loaded from cache
10:17:46  Store 17: loaded from cache
10:17:46  Store 18: loaded from cache
10:17:46  Store 19: loaded from cache
10:17:46  Store 20: loaded from cache
10:17:46  Store 21: loaded from cache
10:17:46  Store 22: loaded from cache
10:17:46  Store 23: loaded from cache
10:17:46  Store 24: loaded from cache
10:17:46  Store 25: loaded from cache
10:17:46  Store 26

**Evaluation**

In [34]:
# df_lgbm= pd.read_csv(os.path.join(RESULTS_DIR, "lgbm_predictions.csv"))
# df_stat_fc= pd.read_csv(os.path.join(RESULTS_DIR, "statistical_predictions.csv"))
# df_sbc= pd.read_csv(os.path.join(RESULTS_DIR, "sbc_classification.csv"))
# print(df_lgbm.head(), df_stat_fc.head(), df_sbc.head())

In [35]:
###### metrics — matching Andrade & Cunha (2023) Article_8_Results.ipynb ######
# MAE / RMSE / sMAPE and their std : original unit_sales scale
# MASE / R2                        : log1p scale (paper computes with expm1 commented out in their code)
# NWRMSLE                          : our addition from kaggle competition - log1p with perishable weights



def compute_metrics(actual, pred, weight):
    a = np.array(actual, dtype=float)
    p = np.clip(np.array(pred, dtype=float), 0, None)
    w = np.array(weight, dtype=float)

    ##### original units (from Article 8  notebook) ####
    ae = np.abs(a - p)
    se = (a - p) ** 2

    mae      = float(np.mean(ae))
    std_mae  = float(pd.Series(ae).std())            # pandas .std(), ddof=1
    rmse     = float(np.sqrt(np.mean(se)))
    std_rmse = float(np.sqrt(pd.Series(se).std()))   # sqrt(std(SE)) — cell 18

    #### paper calls this sMAPE but computes a bounded MAPE (from article 8 notebook):
    #   0 if actual == 0, else min(|error| / |actual|, 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        ape = np.where(a != 0, np.minimum(ae / np.abs(a), 1.0), 0.0)
    smape     = float(np.mean(ape) * 100)
    std_smape = float(pd.Series(ape).std() * 100)

    ##### log1p scale (from article 8 notebook)#####
    al = np.log1p(a)
    pl = np.log1p(p)

    mae_log = float(np.mean(np.abs(al - pl)))

    # MASE with global naive denominator (as in article 8 notebook) — computed across the whole
    # concatenated frame, so naive pairs may cross series boundaries
    naive_mae = float(np.mean(np.abs(al[1:] - al[:-1])))
    mase      = mae_log / naive_mae if naive_mae > 0 else float("nan")
    # paper's notebooks do not compute Std MASE; this is our formula
    std_mase  = float(pd.Series(np.abs(al - pl) / naive_mae).std()) if naive_mae > 0 else float("nan")

    r2 = float(r2_score(al, pl))

    nwrmsle = float(np.sqrt(np.sum(w * (al - pl) ** 2) / np.sum(w)))

    return {
        "MAE"      : round(mae, 2),
        "Std MAE"  : round(std_mae, 2),
        "RMSE"     : round(rmse, 2),
        "Std RMSE" : round(std_rmse, 2),
        "sMAPE(%)" : round(smape, 2),
        "Std sMAPE": round(std_smape, 2),
        "MASE"     : round(mase, 3),
        "Std MASE" : round(std_mase, 3),
        "R2"       : round(r2, 3),
        "NWRMSLE"  : round(nwrmsle, 5),
    }

# LGBM
lgbm_m = compute_metrics(df_lgbm.actual, df_lgbm.pred, df_lgbm.weight)

# Statistical
df_test_actual        = df_lgbm[["unique_id","store_nbr","item_nbr","horizon","actual","weight"]].copy()
df_stat_fc["horizon"] = df_stat_fc.groupby("unique_id").cumcount() + 1
df_stat_eval = df_test_actual.merge(
    df_stat_fc[["unique_id","horizon","pred"]], on=["unique_id","horizon"], how="left")
df_stat_eval["pred"] = df_stat_eval["pred"].fillna(0).clip(lower=0)
stat_m = compute_metrics(df_stat_eval.actual, df_stat_eval.pred, df_stat_eval.weight)

**Results**

In [36]:
#### results ######################

print("\n" + "=" * 88)
print("LightGBM and Statistical Benchmarks — case study 1 - Andrade & Cunha (2023)")
print(f"Corporación Favorita | Weekly | Test: last {TEST_WEEKS} weeks | h=1..{TEST_WEEKS}")
print(f"Series: {len(df_lgbm.groupby(['store_nbr','item_nbr'])):,}  Stores: {len(stores)}  Items: {len(item_cols):,}")
print("=" * 88)

print(f"\nDemand classification (ADI threshold = {ADI_THRESH}):")
for cat, n in df_sbc.category.value_counts().items():
    mdl = {"smooth": "AutoETS(s=52)", "non-smooth": "CrostonSBA"}[cat]
    print(f"  {cat:<14}: {n:>6,}  ({n/len(df_sbc)*100:5.1f}%)  → {mdl}")

print(f"\n{'Model':<33} {'MAE':>7} {'StdMAE':>7} {'RMSE':>7} {'StdRMSE':>8} "
      f"{'sMAPE':>6} {'StdsMAPE':>9} {'MASE':>6} {'StdMASE':>8} {'R2':>6} {'NWRMSLE':>9}")
print("-" * 108)
print(f"{'Paper: XGBoost':<33} {'18.18':>7} {'77.40':>7} {'79.51':>7} {'665.04':>8} "
      f"{'29.00':>6} {'30.42':>9} {'0.856*':>6} {'0.883*':>8} {'0.856':>6} {'-':>9}")
print(f"{'Paper: BaseLift':<33} {'24.81':>7} {'90.59':>7} {'93.93':>7} {'774.40':>8} "
      f"{'35.84':>6} {'36.80':>9} {'1.168':>6} {'—':>8} {'0.561':>6} {'-':>9}")

for label, m in [("Statistical (smooth/non-smooth)", stat_m),
                 ("ML: LightGBM (per-store)", lgbm_m)]:
    print(f"{label:<33} {m['MAE']:>7} {m['Std MAE']:>7} {m['RMSE']:>7} "
          f"{m['Std RMSE']:>8} {m['sMAPE(%)']:>6} {m['Std sMAPE']:>9} "
          f"{m['MASE']:>6} {m['Std MASE']:>8} {m['R2']:>6} {m['NWRMSLE']:>9}")

print(f"\n* Paper MASE=0.856 is a transcription error — equals R2=0.856 exactly.")
print(f"  MASE and R2 computed in log1p scale. MAE/RMSE/sMAPE in original units.")
print(f"\nNotes:")
print(f"  MAE/RMSE/sMAPE: original unit_sales scale (expm1 applied to log1p predictions)")
print(f"  MASE/R2: log1p scale (consistent with paper's Table 4 computation)")
print(f"  NWRMSLE: perishable weight=1.25, non-perishable=1.0")
print(f"  sMAPE: paper labels it sMAPE but computes bounded MAPE —")
print(f"         0 if actual==0, else min(|error|/|actual|, 1), averaged x100")
print(f"  StdRMSE: sqrt(std(squared errors)), matching Article 8 notebok ")
print(f"  MASE: global naive denominator mean(|a[t]-a[t-1]|) across the full frame,")
print(f"        as in Article 8 notebook — naive pairs cross series boundaries")
print(f"  Std MASE: not computed in the paper's notebooks; shown here as")
print(f"            std of scaled absolute errors")
print(f"  seed=42: paper's XGBoost has no seed - exact replication not possible")
print(f"  LGBM: 54 per-store models matching paper's XGBoost design")
print(f"  Statistical: ADI-based classification — AutoETS(s=52) for smooth (ADI<{ADI_THRESH}),")
print(f"               SBA for non-smooth (ADI≥{ADI_THRESH})")

pd.DataFrame([stat_m | {"model": "Statistical"}, lgbm_m | {"model": "LightGBM"}]).to_csv(
    os.path.join(RESULTS_DIR, "metrics.csv"), index=False)
print(f"\nOutputs: {RESULTS_DIR}")


LightGBM and Statistical Benchmarks — case study 1 - Andrade & Cunha (2023)
Corporación Favorita | Weekly | Test: last 8 weeks | h=1..8
Series: 87,642  Stores: 54  Items: 1,623

Demand classification (ADI threshold = 1.32):
  smooth        : 63,206  ( 72.1%)  → AutoETS(s=52)
  non-smooth    : 24,436  ( 27.9%)  → CrostonSBA

Model                                 MAE  StdMAE    RMSE  StdRMSE  sMAPE  StdsMAPE   MASE  StdMASE     R2   NWRMSLE
------------------------------------------------------------------------------------------------------------
Paper: XGBoost                      18.18   77.40   79.51   665.04  29.00     30.42 0.856*   0.883*  0.856         -
Paper: BaseLift                     24.81   90.59   93.93   774.40  35.84     36.80  1.168        —  0.561         -
Statistical (smooth/non-smooth)     22.13   89.35   92.05   678.76  32.92     35.95  1.039    1.511  0.684   0.98949
ML: LightGBM (per-store)            14.75   71.76   73.26   667.46  27.64     32.58  0.591     0

In [37]:
#check with sl =1 and 52 in StatsForecast AutoETS model
sf1 = StatsForecast(models=[AutoETS(season_length=1)], freq=1, n_jobs=-1)
sf1.fit(df=df_stat_long[df_stat_long.unique_id=="10_1004550"])
print(sf1.fitted_[0,0].model_["method"], sf1.fitted_[0,0].model_["aicc"])

sfk = StatsForecast(models=[AutoETS(season_length=52)], freq=1, n_jobs=-1)
sfk.fit(df=df_stat_long[df_stat_long.unique_id=="10_1004550"])
print(sfk.fitted_[0,0].model_["method"], sfk.fitted_[0,0].model_["aicc"])

ETS(A,N,N) 1460.5053232755022
ETS(A,N,N) 1460.5053232755022
